In [75]:
from bs4 import BeautifulSoup
from selenium import webdriver

In [76]:
url_dog = "https://pethouse.com.vn/cua-hang/cho-canh/"
url_cat = "https://pethouse.com.vn/cua-hang/meo-canh/"

folder_stored_dog = "dog"
folder_stored_cat = "cat"

driver = webdriver.Chrome()
driver.get(url_dog)


In [77]:
def extract_name_link(driver):
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    print(soup.title.text)
    items = soup.find_all('div', class_="product-category col product first")
    names = []
    links = []
    print(len(items))
    for item in items:
        a_tag = item.find('div', class_='col-inner')
        link = a_tag.find('a')['href']
        name = a_tag.find('h5', class_='uppercase header-title').text
        name = name.replace("\n", "").replace("\t", "").strip()
        links.append(link)
        names.append(name)

   
    items_2 = soup.find_all('div', class_='product-category col product')
    print(len(items_2))
    for item in items_2:
        a_tag = item.find('div', class_='col-inner')
        link = a_tag.find('a')['href']
        name = a_tag.find('h5', class_='uppercase header-title').text
        links.append(link)
        name = name.replace("\n", "").replace("\t", "").strip()
        names.append(name)
    items_3 = soup.find_all('div', class_='product-category col product last')
    print(len(items_3))
    for item in items_3:
        a_tag = item.find('div', class_='col-inner')
        link = a_tag.find('a')['href']
        name = a_tag.find('h5', class_='uppercase header-title').text
        links.append(link)
        name = name.replace("\n", "").replace("\t", "").strip()
        names.append(name)
        

    return names, links

In [78]:
# dog have 41 items
names, links = extract_name_link(driver)
print(names[11:13])
print(links[11:13])

Chó cảnh - Pet House - Cửa hàng thú cưng và phụ kiện
11
20
10
['Chó poodle', 'Chó Chihuahua']
['https://pethouse.com.vn/cua-hang/cho-canh/cho-poodle/', 'https://pethouse.com.vn/cua-hang/cho-canh/cho-chihuahua/']


In [79]:
from typing import List, Optional, Tuple
from urllib.parse import urljoin
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

def get_infor_item(driver, link: str, name: str
                  ) -> Tuple[str, List[str], List[str], Optional[str]]:
    """
    Trả về:
      (name, infor_links, infor_names, next_link_or_None)
    """
    driver.get(link)

    # Chờ DOM có ít nhất 1 box ảnh (tùy site, chỉnh selector cho đúng)
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.box-image"))
        )
    except Exception:
        # Không thấy item nào -> trả về rỗng
        return (name, [], [], None)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    infor_links, infor_names = [], []
    for item in soup.select("div.box-image"):
        a = item.find("a")
        if not a:
            continue
        href = a.get("href")
        title = a.get("aria-label") or a.get("title") or a.text.strip()
        if not href:
            continue

        # Chuẩn hoá thành absolute URL
        abs_href = urljoin(link, href)

        # Tránh trùng
        if abs_href in infor_links:
            continue

        infor_links.append(abs_href)
        infor_names.append(title)

    print(f"Total item in {name}: {len(infor_links)}")

    # Tìm trang kế tiếp
    next_link = None
    pagination = soup.find("nav", class_="woocommerce-pagination")
    if pagination:
        # 1) Thử link có rel="next"
        a_next = pagination.find("a", rel="next")
        # 2) Nếu không có, lấy nút có text "Next" hoặc ký hiệu >>
        if not a_next:
            candidates = pagination.select("a.page-number")
            for a in candidates[::-1]:  # duyệt từ cuối
                txt = (a.text or "").strip().lower()
                if txt in {"next", ">", "»"}:
                    a_next = a
                    break
        # 3) Fallback: chọn link có class 'next' hoặc data-page lớn hơn current
        if not a_next:
            a_next = pagination.select_one("a.next, a.next-page")

        if a_next and a_next.get("href"):
            next_link = urljoin(link, a_next["href"])

    return (name, infor_links, infor_names, next_link)


In [80]:
name_topic, link_info, infor_name, next_page = get_infor_item(driver, links[0], names[0])
print(name_topic)
print(link_info)
print(infor_name)
print(next_page)

Total item in Chó phốc sóc: 52
Chó phốc sóc
['https://pethouse.com.vn/san-pham/cho-phoc-soc-mini-mau-vang-cam-ma-ps4131/', 'https://pethouse.com.vn/san-pham/cho-pomeranian-vip-mau-vang-mini-ma-ps4130/', 'https://pethouse.com.vn/san-pham/cho-phoc-soc-mini-mau-kem-ma-ps4100/', 'https://pethouse.com.vn/san-pham/cho-phoc-soc-mini-mau-trang-ma-ps3501/', 'https://pethouse.com.vn/san-pham/cho-pomeranian-mau-choco-ma-ps3001/', 'https://pethouse.com.vn/san-pham/cho-pomeranian-mau-vang-mini-ma-ps2044/', 'https://pethouse.com.vn/san-pham/cho-phoc-soc-mini-vang-cam-vip-ma-ps151/', 'https://pethouse.com.vn/san-pham/cho-phoc-soc-vang-nhat-ma-ps1133/', 'https://pethouse.com.vn/san-pham/cho-phoc-soc-mini-vip-vang-cam-ma-ps3284/', 'https://pethouse.com.vn/san-pham/cho-pomeranian-mau-choco-ma-ps3283/', 'https://pethouse.com.vn/san-pham/cho-phoc-soc-mini-trang-ps3249/', 'https://pethouse.com.vn/san-pham/cho-phoc-soc-mini-mau-choco-ma-ps3293/', 'https://pethouse.com.vn/san-pham/cho-phoc-soc-mau-merle-ma-p

In [81]:
import unicodedata

def strip_diacritics(s: str) -> str:
    s = s.replace('đ', 'd').replace('Đ', 'D')
    d = unicodedata.normalize('NFD', s)
    d = ''.join(ch for ch in d if unicodedata.category(ch) != 'Mn')
    return unicodedata.normalize('NFC', d)

def eq_icase_noaccent(a: str, b: str) -> bool:
    return strip_diacritics(a).casefold() == strip_diacritics(b).casefold()

In [82]:
def get_infor_a_item(driver, link, name, topic):
    driver.get(link)
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    #link_img = soup.find('div', class_='col is-nav-selected first')
    # Lấy tất cả div chứa ảnh gallery của WooCommerce
    img_divs = soup.select('div.woocommerce-product-gallery__image')
    #print(img_divs)
    #print(len(img_divs))
    img_links = []
    for div in img_divs:
        a_tag = div.find('a')      # thẻ <a href="...scaled.jpg">
        if a_tag and a_tag.get('href'):
            href = a_tag['href']

            img_links.append(href)

    # Ví dụ: in ra để kiểm tra
    #print(img_links)
    #print(len(link1))
    #print(link1)
    title = name
    category = topic
    # description_item = soup.find('div', class_='product-footer')
    # descr = description_item.find_all('p', class_="product-footer-paragraph")
    root = soup.select_one(".product-footer .accordion-inner") or soup.select_one(".product-footer")
    if not root:
        return (img_links, title, [], None, None, category)
    items = root.select("p, ol li")

    text = [el.get_text(strip=True) for el in items if el.get_text(strip=True)]
    #print(text)
    
    price_item = soup.find('span', class_='woocommerce-Price-amount amount').text

    more_inf = soup.find('div', class_='product-short-description')
    if not more_inf:
        return (img_links, title, text, price_item, None, category)
    more_infor = more_inf.find_all('td')
    return (img_links, title, text, price_item, more_infor, category)

In [83]:
import string
links_item = link_info[0]
print(infor_name[0])
# Bỏ các dấu câu
text = infor_name[0].translate(str.maketrans('', '', string.punctuation))
print(text)
links_img, title_item, descr_item, price_item, more_infor, category = get_infor_a_item(driver, links_item, text, name_topic)
print(f"Total image link: {len(links_img)}: {links_img}")

print(f"category: {category}")
print(f"title_item: {title_item}")
print(f"descr_item: {descr_item}")
print(f"price_item: {price_item}")
print(f"more_infor: {more_infor}")

Chó Phốc sóc mini màu vàng cam mã PS4131
Chó Phốc sóc mini màu vàng cam mã PS4131
Total image link: 2: ['https://pethouse.com.vn/wp-content/uploads/2025/10/cho-phoc-soc-mini-mau-vang-cam-ma-PS4131.jpg', 'https://pethouse.com.vn/wp-content/uploads/2025/10/cho-phoc-soc-mini-mau-vang-cam-ma-PS4131.jpg']
category: Chó phốc sóc
title_item: Chó Phốc sóc mini màu vàng cam mã PS4131
descr_item: ['Chó Phốc sóc mini màu vàng cam mã PS4131nổi bật với vẻ ngoài nhỏ nhắn nhưng vô cùng kiêu kỳ, như một chú sư tử con thu nhỏ. Toàn thân được bao phủ bởi lớp lông dày, bông xù và mềm mượt, mang sắc vàng cam rực rỡ, càng thêm nổi bật dưới ánh nắng. Khuôn mặt tròn xinh, mũi nhỏ đen nhánh, đôi mắt đen long lanh và tai nhỏ dựng đứng tạo nên vẻ lanh lợi, thông minh. Thân hình gọn gàng, chân ngắn, đuôi cong phủ lông xù vắt gọn trên lưng khiến chúng trông như một quả cầu lửa di động. Phốc sóc mini vàng cam không chỉ đẹp mà còn rất tinh nghịch, thân thiện và tình cảm, luôn thích được bồng bế, vuốt ve và là “ngôi

In [84]:
import json
import requests
import os
import requests

def download_image(url, filename):
    try:
        response = requests.get(url)
        if response.status_code == 200:
            if not os.path.exists(os.path.dirname(filename)):
                os.makedirs(os.path.dirname(filename))

            with open(filename, 'wb') as f:
                f.write(response.content)
    except Exception as e:
        print(f"Error downloading image from {url}: {e}")
        return

def format_data(links_img, title_item, category, descr_item, price_item, more_infor, filename):
    data = {
        "title": title_item,
        "category": category,
        "description": descr_item,
        "price": price_item,
        "more_information": {},
        "link_image_file": links_img
    }
    if data["link_image_file"] == []:
        return data
    if more_infor is None:
        return data
    for i in range(0, len(more_infor), 2):
        key = more_infor[i].text.strip()
        value = more_infor[i+1].text.strip() if (i + 1) < len(more_infor) else ""
        data["more_information"][key] = value

    # for link in links_img:
    #     try:
    #         if not link.startswith('http'):
    #             continue
    #         response = requests.get(link)
    #         if response.status_code == 200:
    #             image_data = response.content
    #             data["images"].append({
    #                 "filename": filename,
    #                 "data": image_data.hex()  # Store image data as hex string
    #             })
    #             #download_image(link, filename)
            
    #     except Exception as e:
    #         print(f"Error downloading image from {link}: {e}")

    return data

In [85]:
data = format_data(links_img, title_item, category, descr_item, price_item, more_infor, "image_item.jpg")
print(data)

{'title': 'Chó Phốc sóc mini màu vàng cam mã PS4131', 'category': 'Chó phốc sóc', 'description': ['Chó Phốc sóc mini màu vàng cam mã PS4131nổi bật với vẻ ngoài nhỏ nhắn nhưng vô cùng kiêu kỳ, như một chú sư tử con thu nhỏ. Toàn thân được bao phủ bởi lớp lông dày, bông xù và mềm mượt, mang sắc vàng cam rực rỡ, càng thêm nổi bật dưới ánh nắng. Khuôn mặt tròn xinh, mũi nhỏ đen nhánh, đôi mắt đen long lanh và tai nhỏ dựng đứng tạo nên vẻ lanh lợi, thông minh. Thân hình gọn gàng, chân ngắn, đuôi cong phủ lông xù vắt gọn trên lưng khiến chúng trông như một quả cầu lửa di động. Phốc sóc mini vàng cam không chỉ đẹp mà còn rất tinh nghịch, thân thiện và tình cảm, luôn thích được bồng bế, vuốt ve và là “ngôi sao nhỏ” trong mọi ánh nhìn.', 'Quyền lợi có được khi muaChó Phốc sóc mini màu vàng cam mã PS4131tại Pet House', 'Bảo hành thuần chủng trọn đời.', 'Bảo hành bệnh truyền nhiễm nguy hiểm ở chó là Care và Parvo 7 ngày đầu về nhà mới. Ngoài ra, quý khách có thể mua thêm gói bảo hiểm sức khỏe 1 n

# Download Full Data

In [86]:
def __main__():
    url_dog = "https://pethouse.com.vn/cua-hang/cho-canh/"
    driver = webdriver.Chrome()
    driver.get(url_dog)
    folder_stored_dog = "dog"
    names, links = extract_name_link(driver)
    data_all = {}
    print(f"Total categories to process: {len(links)}")
    print(names)
    
    print(links)
    print("Starting data collection...")
    
    for i in range(4):
        name_topic, link_info, infor_name, next_page = get_infor_item(driver, links[i], names[i])
        for j in range(len(link_info)):
            text = infor_name[j].translate(str.maketrans('', '', string.punctuation))
            links_img, title_item, descr_item, price_item, more_infor, category = get_infor_a_item(driver, link_info[j], text, name_topic)
            data = format_data(links_img, title_item, category, descr_item, price_item, more_infor, f"{folder_stored_dog}/images/{text}.jpg")
            print(f"Collected data for item: {text}")
            if data["link_image_file"] == []:
                continue
            data_all[text] = data

    out_dir = folder_stored_dog                     # = "dog"
    out_json = os.path.join(out_dir, "data_dog.json")

    # đảm bảo thư mục tồn tại
    os.makedirs(out_dir, exist_ok=True)

    # nếu lỡ có THƯ MỤC trùng tên file -> xoá (hoặc đổi tên tuỳ bạn)
    if os.path.isdir(out_json):

        shutil.rmtree(out_json)


    # bây giờ ghi file bình thường
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(data_all, f, ensure_ascii=False, indent=2)

    print(f"Data collection completed and saved to data_dog.json with size {len(data_all)} items.")
    
    driver.quit()
    
if __name__ == "__main__":
    __main__()



Chó cảnh - Pet House - Cửa hàng thú cưng và phụ kiện
11
20
10
Total categories to process: 41
['Chó phốc sóc', 'Chó Labrador', 'Chó Akita', 'Chó Pug mặt xệ', 'Chó Lạp Xưởng', 'Becgie Bỉ Malinois', 'Chó Border Collie', 'Chó Bắc Kinh', 'Chó Basenji', 'Chó Great Dane', 'Ngao Tây Tạng', 'Chó poodle', 'Chó Chihuahua', 'Chó Alaska', 'Chó Corgi Chân Lùn', 'Chó Shiba', 'Chó Bull Pháp', 'Chó Beagle', 'Chó Doberman', 'Chó Phú Quốc', 'Chó Pitbull', 'Chó Bully', 'Chó Chow Chow', 'Chó Phốc Hươu', 'Chó Hmong Cộc', 'Chó Bichon Frise', 'Chó Bull Anh', 'Chó Bướm Papillon', 'Chó Cocker Spaniel', 'Chó Maltese', 'Chó Yorkshire Terrier', 'Chó Golden', 'Chó Becgie Đức', 'Chó Samoyed', 'Chó Husky', 'Chó Rottweiler', 'Chó Shih Tzu', 'Chó Bắc Hà', 'Boston Terrier', 'Chó Đốm', 'Chó Becgie Nga']
['https://pethouse.com.vn/cua-hang/cho-canh/cho-phoc-soc/', 'https://pethouse.com.vn/cua-hang/cho-canh/cho-labrador/', 'https://pethouse.com.vn/cua-hang/cho-canh/cho-akita/', 'https://pethouse.com.vn/cua-hang/cho-canh/ch

In [87]:
def cat():
    url_cat = "https://pethouse.com.vn/cua-hang/meo-canh/"
    driver = webdriver.Chrome()
    driver.get(url_cat)
    folder_stored_cat = "cat"
    names, links = extract_name_link(driver)
    data_all = {}
    print(f"Total categories to process: {len(links)}")
    print(names)
    
    print(links)
    print("Starting data collection...")
    
    for i in range(5):
        name_topic, link_info, infor_name, next_page = get_infor_item(driver, links[i], names[i])
        for j in range(len(link_info)):
            text = infor_name[j].translate(str.maketrans('', '', string.punctuation))
            links_img, title_item, descr_item, price_item, more_infor, category = get_infor_a_item(driver, link_info[j], text, name_topic)
            data = format_data(links_img, title_item, category, descr_item, price_item, more_infor, f"{folder_stored_cat}/images/{text}.jpg")
            print(f"Collected data for item: {text}")
            if data["link_image_file"] == []:
                continue
            data_all[text] = data

    out_dir = folder_stored_cat                     # = "cat"
    out_json = os.path.join(out_dir, "data_cat.json")

    # đảm bảo thư mục tồn tại
    os.makedirs(out_dir, exist_ok=True)

    # nếu lỡ có THƯ MỤC trùng tên file -> xoá (hoặc đổi tên tuỳ bạn)
    if os.path.isdir(out_json):

        shutil.rmtree(out_json)


    # bây giờ ghi file bình thường
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(data_all, f, ensure_ascii=False, indent=2)

    print(f"Data collection completed and saved to data_dog.json with size {len(data_all)} items.")
    
    driver.quit()
    
if __name__ == "__main__":
    cat()



Mèo cảnh - Pet House - Cửa hàng thú cưng và phụ kiện
4
7
3
Total categories to process: 14
['Mèo anh lông dài', 'Mèo Himalaya', 'Mèo Mỹ Lông Ngắn', 'Mèo Sphynx - Mèo Không Lông', 'Mèo anh lông ngắn', 'Mèo Ba Tư', 'Mèo Maine Coon', 'Mèo Munchkin', 'Mèo Mỹ Tai Xoắn', 'Mèo Ragdoll', 'Mèo Tuxedo', 'Mèo Bengal', 'Mèo Mướp', 'Mèo Scottish Fold']
['https://pethouse.com.vn/cua-hang/meo-canh/meo-anh-long-dai/', 'https://pethouse.com.vn/cua-hang/meo-canh/meo-himalaya/', 'https://pethouse.com.vn/cua-hang/meo-canh/meo-my-long-ngan/', 'https://pethouse.com.vn/cua-hang/meo-canh/meo-sphynx/', 'https://pethouse.com.vn/cua-hang/meo-canh/meo-anh-long-ngan/', 'https://pethouse.com.vn/cua-hang/meo-canh/meo-ba-tu/', 'https://pethouse.com.vn/cua-hang/meo-canh/meo-maine-coon/', 'https://pethouse.com.vn/cua-hang/meo-canh/meo-munchkin/', 'https://pethouse.com.vn/cua-hang/meo-canh/meo-my-tai-xoan/', 'https://pethouse.com.vn/cua-hang/meo-canh/meo-ragdoll/', 'https://pethouse.com.vn/cua-hang/meo-canh/meo-tuxedo/'

In [ ]:
# Collect dog data by function
def dog():
    url_dog = "https://pethouse.com.vn/cua-hang/cho-canh/"
    driver = webdriver.Chrome()
    driver.get(url_dog)
    folder_stored_dog = "dog"
    names, links = extract_name_link(driver)
    data_all = {}
    print(f"Total categories to process: {len(links)}")
    print(names)
    
    print(links)
    print("Starting data collection...")
    
    for i in range(4):
        name_topic, link_info, infor_name, next_page = get_infor_item(driver, links[i], names[i])
        for j in range(len(link_info)):
            text = infor_name[j].translate(str.maketrans('', '', string.punctuation))
            links_img, title_item, descr_item, price_item, more_infor, category = get_infor_a_item(driver, link_info[j], text, name_topic)
            data = format_data(links_img, title_item, category, descr_item, price_item, more_infor, f"{folder_stored_dog}/images/{text}.jpg")
            print(f"Collected data for item: {text}")
            if data["link_image_file"] == []:
                continue
            data_all[text] = data

    out_dir = folder_stored_dog                     # = "dog"
    out_json = os.path.join(out_dir, "data_dog.json")

    # đảm bảo thư mục tồn tại
    os.makedirs(out_dir, exist_ok=True)

    # nếu lỡ có THƯ MỤC trùng tên file -> xoá (hoặc đổi tên tuỳ bạn)
    if os.path.isdir(out_json):

        shutil.rmtree(out_json)


    # bây giờ ghi file bình thường
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(data_all, f, ensure_ascii=False, indent=2)

    print(f"Data collection completed and saved to data_dog.json with size {len(data_all)} items.")
    
    driver.quit()


In [ ]:
print("Dog data collection started...")
dog()
print("Dog data collection finished.")